In [1]:
import numpy as np

file_structure = {
    "data": np.float32,   # [H, W]
    "mask": np.uint8,     # [H, W]
    "valid": np.uint8,    # [H, W]
    "meta": {
        "height": int,
        "width": int,
        "nodata_fill_value": float,
        "created_from": {
            "data_path": str,
            "mask_path": str
        }
    }
}

no_data_value = -255

# RAW DEM Data

In [2]:
import numpy as np

dem_path = "/home/nikitachernysh/Storage/Projects/lidar-archaeology-segmentation/data/raw/DEM.npz"
mask_path = "/home/nikitachernysh/Storage/Projects/lidar-archaeology-segmentation/data/raw/Mounds_raster_mask_opened_closed.npy"

data = np.load(dem_path)
dem = data["dataset"].astype(np.float32)
valid = data["validMask"].astype(bool)
mask = np.load(mask_path).astype(np.uint8)

f_name = "DEM_raw"

In [3]:
dem[~valid] = no_data_value

In [4]:
assert dem.shape == valid.shape
assert dem.shape == mask.shape
assert valid.shape == mask.shape

In [5]:
file_structure["data"] = np.stack([dem, dem, dem], axis=0)
file_structure["mask"] = mask
file_structure["valid"] = valid

file_structure["meta"]["height"] = dem.shape[0]
file_structure["meta"]["width"] = dem.shape[1]
file_structure["meta"]["nodata_fill_value"] = no_data_value

file_structure["meta"]["created_from"]["data_path"] = dem_path
file_structure["meta"]["created_from"]["mask_path"] = mask_path

In [6]:
np.savez_compressed(f"../data/processed/{f_name}.npz", **file_structure)

# RAW DEM Dataset
- raw DEM
- confidence shadow mask

In [7]:
import numpy as np

data_path = "/home/nikitachernysh/Storage/Projects/lidar-archaeology-segmentation/data/processed/DEM_raw.npz"

data = np.load(data_path)
dem = data["data"].astype(np.float32)
valid = data["mask"].astype(np.uint8)
mask = data["valid"].astype(np.uint8)

f_name = "DEM_raw_shadowed"

In [8]:
assert dem.shape == valid.shape
assert dem.shape == mask.shape
assert valid.shape == mask.shape

In [9]:
import numpy as np
from scipy.ndimage import distance_transform_edt

# input: mask -> 2D numpy array, binary labeling mask

resolution = 0.5  # meters per pixel

# paper values in meters
shadow_len_m = 10  # or 4

shadow_len_px = shadow_len_m / resolution

mask = (mask > 0).astype(np.uint8)

dist_outside = distance_transform_edt(mask == 0)

confidence_mask = np.where(
    mask == 1,
    1.0,
    np.clip(1.0 - dist_outside / shadow_len_px, 0.0, 1.0)  # pyright: ignore[reportOperatorIssue]
).astype(np.float32)

In [10]:
file_structure["data"] = np.stack([dem, dem, dem], axis=0)
file_structure["mask"] = confidence_mask
file_structure["valid"] = valid

file_structure["meta"]["height"] = dem.shape[0]
file_structure["meta"]["width"] = dem.shape[1]
file_structure["meta"]["nodata_fill_value"] = no_data_value

file_structure["meta"]["created_from"]["data_path"] = data_path
file_structure["meta"]["created_from"]["mask_path"] = data_path

In [11]:
np.savez_compressed(f"../data/processed/{f_name}.npz", **file_structure)